# Lecture 3 self-check: Profiling, algorithms and optimisation (with answers)

Twelve short questions on the main points of this lecture. The quiz is not assessed: it is a check that you have the main ideas before you move on. The code in each question is complete as shown, and runs in a notebook in the module's `mpm2026` environment. The answer to each question, with an explanation, is under *Show answer* below its options.

Run the code cell below first (or use *Run All*). Then click one option under each question. Your choices are listed at the end of the notebook. They are kept until you restart the kernel, so note them down if you want to keep them.

In [ ]:
# Run this cell first. It sets up the clickable options under each question.
from IPython.display import display
from ipywidgets import Layout, RadioButtons

my_answers = {}


def choose(question, letters):
    """Show clickable options for one question and remember your choice."""
    buttons = RadioButtons(options=list(letters), value=None, layout=Layout(width="auto"))
    buttons.observe(lambda change: my_answers.__setitem__(question, change["new"]), names="value")
    display(buttons)

### Question 1

`time.time()` measures wall time and `time.process_time()` measures CPU time. What does this code print, to one decimal place?

```python
import time

start_wall, start_cpu = time.time(), time.process_time()
time.sleep(0.5)
wall = time.time() - start_wall
cpu = time.process_time() - start_cpu

print(round(wall, 1), round(cpu, 1))
```

- **(A)** `0.5 0.5`
- **(B)** `0.5 0.0`
- **(C)** `0.0 0.0`
- **(D)** `0.0 0.5`


In [ ]:
choose(1, "ABCD")

<details>
<summary>Show answer</summary>

**Answer: (B)**

`time.sleep(0.5)` makes the program wait for half a second. Wall time is what a clock on the wall measures, from start to end, including any waiting, so it is about 0.5 s. CPU time counts only the time the processor spends working on our program, and while it sleeps it does no work, so the CPU time is almost zero. Waiting for a file or a web page (as in the lecture's `requests.get` example) works the same way. CPU time is larger than wall time only when several threads of our program work in parallel on different cores.

</details>

### Question 2

One notebook cell runs

```python
calls = [0]


def f():
    calls[0] += 1
```

and the next cell runs `%timeit -r 7 -n 100 f()`. What is `calls[0]` after that?

- **(A)** `7`
- **(B)** `100`
- **(C)** `107`
- **(D)** `700`


In [ ]:
choose(2, "ABCD")

<details>
<summary>Show answer</summary>

**Answer: (D)**

`%timeit` repeats the measurement `-r` times (runs), and in each run it executes the statement `-n` times (loops), so `f()` is called 7 × 100 = 700 times. It reports the mean and standard deviation of the time per loop over the 7 runs. Without `-n`, `%timeit` first works out a number of loops itself, which calls `f()` many more times.

</details>

### Question 3

```python
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n - 1)
```

What happens when you call `factorial(5000)` with Python's default settings?

- **(A)** It raises a `MemoryError`
- **(B)** It returns 5000!, an integer with 16,326 digits
- **(C)** It raises a `RecursionError`
- **(D)** It raises an `OverflowError`


In [ ]:
choose(3, "ABCD")

<details>
<summary>Show answer</summary>

**Answer: (C)**

Each call that hasn't returned yet keeps a frame on the call stack, and `factorial(5000)` needs about 5000 of them at once. Python limits the depth to about 1000 by default (`sys.getrecursionlimit()`), and going deeper raises a `RecursionError`, Python's guard against a stack overflow. Python integers have no maximum size, so the result itself would be no problem: a version that uses a `for` loop instead returns it. The call stack runs out long before the memory does.

</details>

### Question 4

```python
def tidy(value):
    return round(value, 2)


def simulate(n):
    total = 0.0
    for i in range(n):
        total += (i % 7) * 0.5
    return total


def main():
    results = [tidy(i / 3) for i in range(1_000)]
    results.append(simulate(2_000_000))
    return results
```

In the table that `%prun main()` prints, which of the three functions has the largest `tottime`?

- **(A)** `main`
- **(B)** `tidy`
- **(C)** `simulate`


In [ ]:
choose(4, "ABC")

<details>
<summary>Show answer</summary>

**Answer: (C)**

`tottime` is the time spent in a function's own code, *excluding* the functions it calls. `simulate` runs a loop of 2,000,000 iterations, so it has by far the largest `tottime`. `main` has the largest `cumtime`, which includes the time in everything it calls (`simulate` and `tidy`), but its own code takes very little. `tidy` has the largest `ncalls` (1,000), but each call is quick. To find the function to optimise, sort by `tottime`.

</details>

### Question 5

```python
def grid_sum(n):
    total = 0
    for i in range(n):
        for j in range(n):
            total += i * j
    return total
```

In the output of `%lprun -f grid_sum grid_sum(100)`, what is the number in the `Hits` column for the line `total += i * j`?

- **(A)** `1`
- **(B)** `100`
- **(C)** `200`
- **(D)** `10000`


In [ ]:
choose(5, "ABCD")

<details>
<summary>Show answer</summary>

**Answer: (D)**

`Hits` is the number of times a line was executed. The line `total += i * j` is inside two nested loops of 100 iterations each, so it runs 100 × 100 = 10,000 times. The line of the outer loop runs about 100 times, the line of the inner loop about 100 times per outer iteration, and `total = 0` once. Lines with many hits, or with a large time per hit, are the ones worth optimising.

</details>

### Question 6

```python
def simulate(n):
    total = 0
    for i in range(n):
        total += i % 7
    return total


def run():
    data = list(range(1_000))
    result = simulate(100_000)
    return len(data) + result
```

With `line_profiler` loaded, you run

```python
%lprun -f run run()
```

For which lines does the output give the hits and the time?

- **(A)** The lines of `simulate` only
- **(B)** The lines of `run` only
- **(C)** The lines of `run` and the lines of `simulate`


In [ ]:
choose(6, "ABC")

<details>
<summary>Show answer</summary>

**Answer: (B)**

`%lprun -f <function> <call>` runs `<call>` and times the lines of each function named with `-f`, and only those. Here that is `run`, so the output lists the four lines of `run`. The line `result = simulate(100_000)` appears with the total time of the call, but the lines inside `simulate` are not shown. To see them as well, name it too: `%lprun -f run -f simulate run()`, or just `%lprun -f simulate run()`. `%lprun` never picks the slow function for you: `%prun` tells you which function to name.

</details>

### Question 7

```python
def unique(values):
    seen = []
    for v in values:
        if v not in seen:
            seen.append(v)
    return seen
```

`values` is a list of $n$ different numbers. What is the time complexity of `unique(values)`?

- **(A)** $O(n^2)$
- **(B)** $O(n)$
- **(C)** $O(n \log n)$
- **(D)** $O(n^3)$


In [ ]:
choose(7, "ABCD")

<details>
<summary>Show answer</summary>

**Answer: (A)**

The `for` loop runs $n$ times, and in each iteration `v not in seen` compares `v` with every element of the list `seen`, which grows to $n$ elements: $0 + 1 + \dots + (n - 1) = n(n - 1)/2$ comparisons, which is $O(n^2)$. `in` on a list is a hidden loop. `seen.append(v)` takes constant time, so it doesn't add a factor of $n$. With a set, `seen = set()` and `seen.add(v)`, each `in` check takes constant time on average, and the function becomes $O(n)$.

</details>

### Question 8

A function's wall time $t$ is measured for four input sizes $n$:

| $n$ | 1000 | 2000 | 4000 | 8000 |
|---|---|---|---|---|
| $t$ (s) | 0.02 | 0.08 | 0.32 | 1.28 |

About how long will it take for $n$ = 40,000?

- **(A)** About 25.6 s
- **(B)** About 32 s
- **(C)** About 6.4 s
- **(D)** About 800 s


In [ ]:
choose(8, "ABCD")

<details>
<summary>Show answer</summary>

**Answer: (B)**

Each time $n$ doubles, $t$ is multiplied by 4 = $2^2$, so $t \approx C n^2$: on a log-log plot the points lie on a straight line with slope 2, and the function is $O(n^2)$. Going from $n$ = 8000 to 40,000 multiplies $n$ by 5, so $t$ is multiplied by $5^2 = 25$: 1.28 s × 25 = 32 s. Scaling linearly (×5) would give 6.4 s, as if the slope were 1. Multiplying the factor per doubling by the size ratio (4 × 5 = 20) gives 25.6 s, but a factor of 4 per doubling means $t \propto n^2$, so a factor of 5 in $n$ gives $5^2$. And the factor of 4 per doubling is $2^k$, so the slope $k$ is 2, not 4: a slope of 4 (×625) would give 800 s.

</details>

### Question 9

`x` and `y` are lists of $n$ numbers each. Each version of `common` returns the elements of `x` that are also in `y`.

Version 1:

```python
def common(x, y):
    ys = set(y)
    return [v for v in x if v in ys]
```

Version 2:

```python
def common(x, y):
    return [v for v in x if v in y]
```

Version 3:

```python
def common(x, y):
    return [v for v in set(x) if v in y]
```

Version 4:

```python
def common(x, y):
    ys = sorted(y)
    return [v for v in x if v in ys]
```

Which version takes $O(n)$ time (on average)?

- **(A)** Version 1
- **(B)** Version 4
- **(C)** Version 3
- **(D)** Version 2


In [ ]:
choose(9, "ABCD")

<details>
<summary>Show answer</summary>

**Answer: (A)**

Checking `v in` a list scans the list, which takes $O(n)$ time, so doing it for each of the $n$ values of `x` takes $O(n^2)$ (version 2). Checking `v in` a set takes constant time on average, because a set finds an element from its hash. Building `set(y)` takes $O(n)$ once, so version 1 is $O(n)$. Making a set of `x` (version 3) doesn't help, because the `in` check is still on the list `y`. Sorting `y` (version 4) doesn't help either: `in` doesn't know the list is sorted and still scans it (a binary search would have to be written explicitly), and the sort adds $O(n \log n)$.

</details>

### Question 10

```python
import functools

calls = [0]


@functools.cache
def fib(n):
    calls[0] += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)
```

`fib(20)` is called, and then `fib(20)` is called again. How many times is the body of `fib` executed (how much does `calls[0]` go up) during the first call, and during the second?

- **(A)** 21891, then 21891
- **(B)** 21891, then 0
- **(C)** 21, then 0
- **(D)** 21, then 21


In [ ]:
choose(10, "ABCD")

<details>
<summary>Show answer</summary>

**Answer: (C)**

`functools.cache` stores the result for each argument the first time it is computed, and returns the stored result after that without running the body. During the first call, `fib(20)` needs `fib(19)` and `fib(18)`, and so on down; each of `fib(0)`, ..., `fib(20)` is computed once, so the body runs 21 times (without the cache it would run 21,891 times, because the same values are recomputed over and over). The cache is kept between calls, so the second `fib(20)` is a single lookup and the body doesn't run at all.

</details>

### Question 11

```python
import numpy as np

small = np.array([100, 200, 300]).astype("uint8")
print(small.tolist())
```

What happens?

- **(A)** It prints `[100, 200, 255]`
- **(B)** It prints `[100, 200, 300]`
- **(C)** It raises an `OverflowError`
- **(D)** It prints `[100, 200, 44]`


In [ ]:
choose(11, "ABCD")

<details>
<summary>Show answer</summary>

**Answer: (D)**

`uint8` holds the integers 0 to 255. `astype` converts the values without checking that they fit. 300 doesn't fit, so it wraps around: only its lowest 8 bits are kept, 300 − 256 = 44. No error or warning is raised, and the value is not clipped to 255. Before downcasting, check the data's `min()` and `max()` against the range of the smaller type (`np.iinfo(np.uint8)`).

</details>

### Question 12

What does this print?

```python
doubled = (2 * i for i in range(3))
first = sum(doubled)
second = sum(doubled)
print(first, second)
```

- **(A)** `6 0`
- **(B)** `6 6`
- **(C)** Nothing: the second `sum` raises a `StopIteration`
- **(D)** Nothing: the first `sum` raises a `TypeError`


In [ ]:
choose(12, "ABCD")

<details>
<summary>Show answer</summary>

**Answer: (A)**

`doubled` is a generator: it produces the values 0, 2 and 4 one at a time, as they are asked for, and doesn't store them. The first `sum` takes all three, giving 6, and leaves the generator exhausted. The second `sum` gets no values and returns 0. Iterating over an exhausted generator, with `sum` or a `for` loop, simply stops; only calling `next(doubled)` directly raises a `StopIteration`. `sum` accepts any iterable, generators included. To use the values twice, make a list, or a new generator.

</details>

## Your answers

In [ ]:
for q in range(1, 13):
    print(f"Question {q}: {my_answers.get(q) or 'not answered'}")